In [1]:
with open("../results/2026_deepseek_response/merge.log", "r") as f:
    log = f.read()

In [2]:
inputs = []
paper_path = []
for seg in log.split("============================================================")[1:]:
    if "PAPERS_DIR=/home/wg25r/split_review/datasets/iclr2026_new/papers" in seg:
        continue
    inputs.append(seg.split("--- Merged Inputs ---")[-1].split("--- Merged Review ---")[0].strip())
    paper_path.append(seg.split("--- Merged Inputs ---")[0].split("Paper: ")[-1].split("\n")[0]) 

In [10]:
import sys
sys.path.append("../code")
from agents.model_settings import ModelSettings

from agents import Agent, OpenAIChatCompletionsModel, OpenAIResponsesModel, Runner, function_tool
from pydantic import BaseModel
from paths import prompt_path
import time
from tools import CALIBRATION_REVIEW_DIR, read_file, read_file_full, grep_file, search_file, _search_file_impl  
from openai import AsyncOpenAI
from agents import set_default_openai_client, set_tracing_export_api_key
import os

custom_client = AsyncOpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.getenv("OPENROUTER_API_KEY"))
set_default_openai_client(custom_client)
set_tracing_export_api_key("")

with open(prompt_path("timeline.md"), "r") as f:
    timeline = f.read().replace("{{CURRENT_DATE}}", time.strftime("%Y-%m-%d"))
PAPER_ACCESS_FILE = "The paper path is provided in the user message. Use read_file to read the paper (it reads the whole file by default — do not pass start_line/end_line unless you specifically need a slice) and verify reviewer claims directly."


def load_prompts(path, paper_access, no_cal: bool = False):
    with open(prompt_path(path), "r") as f:
        raw_lines = f.readlines()
    kept_lines = []
    for lineno, line in enumerate(raw_lines, start=1):
        if line.lstrip().startswith("&&"):
            print(f"WARNING: ignoring commented line {lineno} in prompts/{path}: {line.rstrip()}")
            continue
        kept_lines.append(line)
    content = "".join(kept_lines)
    content = content.replace("{{PAPER_ACCESS_INSTRUCTION}}", paper_access)
    cal_instruction = 'You should use tools (like `calibration_search`) to look for paper with similar content or weaknesses and return a list of related papers at the end in a JSON like within XML tag like <related>["RT5SlprCmc", "e9JphzQ5Gr", ...]</related> in a section called # Related Reviews..'
    content = content.replace("{{CALIBRATION_INSTRUCTION}}", cal_instruction)
    return content + "\n\n" + timeline



_merger_instructions = load_prompts("merger.md", paper_access=PAPER_ACCESS_FILE, no_cal=True)

@function_tool
def draft_review(draft: str) -> str:
    """Record the merger's post-filtering draft before calibration or final writing."""
    return "draft recorded"


class CalibrationQuery(BaseModel):
    query: str
    n: int = 4
    low_score: float = -1.0
    high_score: float | None = 11.0

@function_tool
def calibration_search(queries: list[CalibrationQuery]) -> str:
    """RAG retrieval over the human-review corpus.

    Pass a batch of queries; each runs vector search and returns top-n
    hits with avg human score and first 1000 chars. Up to 3 calls
    total across the session (bracket → narrow → optional re-narrow);
    see the calibration protocol in the system prompt for when to use
    each round.

    Args:
        queries: list of {query: str, n?: int, low_score?: float,
            high_score?: float}.
    """
    if not isinstance(queries, list) or not queries:
        raise ValueError("calibration_search: 'queries' must be a non-empty list of query objects.")
    sections = []
    for i, q in enumerate(queries, 1):
        qtext = q.query
        n = q.n
        low_score = q.low_score
        high_score = 11.0 if q.high_score is None else q.high_score
        body = _search_file_impl(qtext, n, "vector", low_score, high_score)
        sections.append(
            f"### Query {i}: {qtext!r}  (n={n}, score=({low_score}, {high_score}))\n{body}"
        )
    return "\n\n".join(sections)
_MODEL_SETTINGS = ModelSettings(extra_body={"effort": "high", "provider": {"only": ["deepseek"]}})
_merger_tools = [read_file, grep_file, calibration_search]
merger = Agent(
    name="Merger",
    instructions=_merger_instructions,
    model="deepseek-v4-flash",
    tools=_merger_tools,
    model_settings=_MODEL_SETTINGS,
)


In [11]:
from tools import CALIBRATION_REVIEW_DIR

In [12]:
MAX_RETRIES = 5
RETRY_DELAY = 3  # seconds
import asyncio
async def run_agent_with_retry(agent, prompt: str, max_turns: int = 30) -> tuple[str, object]:
    agent_name = agent.name
    print(f"  [{agent_name}] starting ...")
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            result = await Runner.run(agent, prompt, max_turns=max_turns)
            output = result.final_output
            if not output or not output.strip():
                if attempt < MAX_RETRIES:
                    print(f"  [{agent_name}] empty response (attempt {attempt}/{MAX_RETRIES}), retrying ...")
                    await asyncio.sleep(RETRY_DELAY + attempt * 5)
                    continue
                raise RuntimeError(f"[{agent_name}] empty response after {MAX_RETRIES} attempts")
            print(f"  [{agent_name}] done")
            return output, result.context_wrapper.usage
        except Exception as e:
            if attempt < MAX_RETRIES:
                wait = RETRY_DELAY * attempt
                print(f"  [{agent_name}] error (attempt {attempt}/{MAX_RETRIES}), waiting {wait}s ... {e}")
                await asyncio.sleep(wait)
            else:
                raise RuntimeError(f"[{agent_name}] {e}") from e
    raise RuntimeError(f"[{agent_name}] failed after {MAX_RETRIES} attempts")


In [55]:
from pathlib import Path
paper_path_abs = paper_path[0]
from tools import allow_path
allow_path(str(Path(paper_path_abs).parent))
start_time = time.monotonic() 
merger_prompt = (
    f"Here is the paper being reviewed (extracted from PDF — formatting "
    f"artifacts are parser issues, not paper problems).\n\n"
    f"Paper path: {paper_path_abs} — use read_file (which reads the whole file by default; do not pass start_line/end_line unless you specifically need a slice) or grep_file to read it.\n\n"
    f"This paper got rated by human with 0/10, 4/10, 2/10 and rejected."
    f"Human reviews directory (for calibration): {CALIBRATION_REVIEW_DIR}\n\n" 
    f"Here are the inputs:{inputs[0]}"
    f"Now produce the final consolidated review following your instructions. "
    f"Remember: many of the harsh critic's points may be nonsensical or overly "
    f"picky — cross-check everything against the actual paper before including it."
)
merged_review, merger_usage = await run_agent_with_retry(merger, merger_prompt)


  [Merger] starting ...
  [read_file] Request to read '/home/wg25r/split_review/datasets/iclr2026_new/papers/Iq1fNZus2W.txt' lines 1 to EOF
  [grep_file] Request to grep for pattern 'condition overhead' in '/home/wg25r/split_review/datasets/iclr2026_new/papers/Iq1fNZus2W.txt'
  [grep_file] Request to grep for pattern 'keyword' in '/home/wg25r/split_review/datasets/iclr2026_new/papers/Iq1fNZus2W.txt'
  [search_file] query='efficient multi-condition control diffusion transformers attention sparsity' mode='vector' n=5 score=(-1.0, 11.0)
  [search_file] query='position-aligned attention keyword-scoped attention DiT' mode='vector' n=5 score=(-1.0, 11.0)
  [search_file] query='early timestep sampling diffusion training convergence' mode='vector' n=5 score=(-1.0, 11.0)
  [search_file] query='multi-condition control diffusion transformer efficiency speedup' mode='vector' n=5 score=(-1.0, 11.0)
  [search_file] query='attention sparsity diffusion transformer position aligned keyword aware' mode=

In [57]:
print(merged_review)

Now I have all the information needed. Let me produce the final consolidated review.

## Summary

This paper addresses the computational bottleneck in multi-condition Diffusion Transformers, where the "concatenate-and-attend" strategy causes attention complexity to scale quadratically with the number of conditions. The authors propose Patch-wise and Keyword-Aware Attention (PKA), which decomposes full attention into two specialized modules: Position-Aligned Attention (PAA) for spatial conditions (one-to-one attention along aligned patches) and Keyword-Scoped Attention (KSA) for subject conditions (attention restricted to keyword-activated regions via a mask). A condition KV cache and an early-timestep training sampling strategy are also introduced. Experiments on FLUX.1 with LoRA fine-tuning report up to 10× inference speedup and 5.12× VRAM reduction while maintaining or improving quality versus OminiControl2 and UniCombine.

## Strengths

- **Well-motivated sparsity analysis.** Figure

In [21]:
prompt = """
You will be given two reviews for two different papers, you need to decide which paper is better, the first one (return -1), the second one (return 1) or they are about the same level (return 0). Note that you are not comparing the reviews, you are comparing the papers based on the reviews. The review tone may not reflect the actual quality of the paper, so you need to read the reviews carefully and understand the content of reviews.
"""

In [30]:
paper_path_abs

'/home/wg25r/split_review/datasets/iclr2026_new/papers/../papers/Iq1fNZus2W.txt'

In [58]:
import re


def remove_scores_and_ratings(
    text: str,
) -> None:

    text = re.sub(r"(?m)^- (?:Avg Score|Scores):.*\n", "", text)

    text = re.sub(r"(?m)^- Decision:.*\n", "", text)

    rating_sections = (
        "Rating",
        "Rating Number",
        "Confidence",
        "Soundness",
        "Presentation", 
        "Contribution",
    )
    names = "|".join(re.escape(name) for name in rating_sections)
    text = re.sub(
        rf"(?ms)^### (?:{names})\n.*?(?=^### |^## |^---$|\Z)",
        "",
        text,
    )

    return text

In [59]:
from openai import OpenAI
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.getenv("OPENROUTER_API_KEY"))
# First API call with reasoning
with open(os.path.join(CALIBRATION_REVIEW_DIR, "0e2pcSxQJS.md"), "r") as f:
    compared_to = remove_scores_and_ratings(f.read())

In [ ]:
response = client.chat.completions.create(
  model="deepseek/deepseek-v4-flash",
  messages=[
          {
            "role": "system",
            "content": prompt
          },
          {
              "role": "user",
              "content": f"Review 1: {compared_to}\n\nReview 2: {merged_review.split('## Score and Decision')[0]}"
          }
        ],
  extra_body={"reasoning": {"enabled": True}}
)

In [69]:
print(response.choices[0].message.reasoning)

We need to decide which paper is better based on the reviews. We are not comparing the reviews themselves, but the papers based on the reviews. The review tone may not reflect actual quality. We need to read the reviews carefully and understand the content of the papers.

Paper 1 (Review 1): PN-GAIL: Leveraging Non-optimal Information from Imperfect Demonstrations. This is about imitation learning from imperfect demonstrations using positive-negative risk and balanced semi-confidence classification. The reviews (4 human reviewers) are mixed but generally positive. However, there are several weaknesses pointed out: empirical results not impressive (performance difference small), reliance on confidence scores, missing comparisons, some theoretical bounds may be loose or rely on unknown variances. One reviewer disagrees with a fundamental motivation claim. Another notes that although based on prior work, it broadens scope. Some minor issues. Overall, the paper seems decent but not groundb